# 05j-o — Fresh test rigenerativo preregistrato

Il robust gate 05j-n ha autorizzato l'apertura del piano fresh-test congelato in 05j-m. Questo notebook genera tutti i 32 pair teacher e valuta i tre checkpoint congelati senza selezione, tuning o retraining. Un pass autorizza soltanto il micro-rollout 05k.

## 1. Checkout riproducibile e teacher canonico

In [ ]:
import importlib,json,os,shutil,subprocess,sys,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';ELM_REF=os.environ.get('HAYFLOW_ELM_REF','main');TEACHER_REPOSITORY='https://github.com/SelfishGene/neuron_as_deep_net.git';TEACHER_COMMIT='074c4666300a8ad246601dab179a97a6942f0f29'
ROOT=Path('/kaggle/working');WORKSPACE=ROOT/'hayflow_workspace';WORKSPACE.mkdir(parents=True,exist_ok=True);ELM_REPO=WORKSPACE/'elmneuron'
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO)
TEACHER_REPO=Path(os.environ.get('HAYFLOW_TEACHER_REPO',WORKSPACE/'neuron_as_deep_net')).expanduser().resolve()
if not (TEACHER_REPO/'.git').is_dir():run(['git','clone',TEACHER_REPOSITORY,TEACHER_REPO])
run(['git','checkout','--detach',TEACHER_COMMIT],cwd=TEACHER_REPO);assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=TEACHER_REPO,text=True).strip()==TEACHER_COMMIT
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();print({'owned':str(ELM_REPO),'teacher':str(TEACHER_REPO),'revision':REVISION})

## 2. Dipendenze, MOD originali e GPU

In [ ]:
run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7','numpy','pandas','matplotlib','h5py','pyarrow','pyyaml'])
SIMULATION_DIR=TEACHER_REPO/'L5PC_NEURON_simulation'
if not list(SIMULATION_DIR.rglob('libnrnmech.so')):run([shutil.which('nrnivmodl') or str(Path(sys.executable).parent/'nrnivmodl'),'mods'],cwd=SIMULATION_DIR)
assert list(SIMULATION_DIR.rglob('libnrnmech.so'));sys.path.insert(0,str(ELM_REPO));importlib.invalidate_caches()
import h5py,numpy as np,pandas as pd,pyarrow,torch,yaml
assert torch.cuda.is_available(),'Attiva una GPU Kaggle: NEURON usa CPU, l’inferenza 05j-n usa GPU.'
print({'torch':torch.__version__,'cuda':torch.cuda.get_device_name(0),'teacher_mods':'ready'})

## 3. Input esatti e catena di provenienza

In [ ]:
import hashlib
INPUT_ROOT=Path('/kaggle/input')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_size';stamp=str(source.stat().st_size)
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
def index_matches(path,expected):
 path=Path(path)
 try:
  if path.is_file():
   with zipfile.ZipFile(path) as archive:
    names=[n for n in archive.namelist() if n.replace('\\','/').endswith('artifact_index.json')];return len(names)==1 and hashlib.sha256(archive.read(names[0])).hexdigest()==expected
  return any(hashlib.sha256(p.read_bytes()).hexdigest()==expected for p in path.rglob('artifact_index.json'))
 except (OSError,zipfile.BadZipFile):return False
def artifact(env,name,marker,expected):
 candidates=([Path(os.environ[env]).expanduser()] if os.environ.get(env) else [])+list(INPUT_ROOT.rglob(name))+[p.parent for p in INPUT_ROOT.rglob(marker)];valid=[p.resolve() for p in candidates if p.exists() and index_matches(p,expected)];assert valid,f'{name} non trovato o incompatibile: {[str(p) for p in candidates]}';return valid[0]
def marker_artifact(name,marker):
 candidates=list(INPUT_ROOT.rglob(name))+[p.parent for p in INPUT_ROOT.rglob(marker)];found=next((p.resolve() for p in candidates if p.exists()),None);assert found is not None,f'{name} non trovato.';return found
def valid_base(path):return (Path(path)/'transition_dataset.h5').is_file() and (Path(path)/'targeted_pilot'/'candidate_trials.parquet').is_file()
base_candidates=([Path(os.environ['HAYFLOW_BASE_DATASET']).expanduser()] if os.environ.get('HAYFLOW_BASE_DATASET') else [])+[p.parent.parent for p in INPUT_ROOT.rglob('candidate_trials.parquet') if p.parent.name=='targeted_pilot'];BASE_DATASET=next((p.resolve() for p in base_candidates if valid_base(p)),None);assert BASE_DATASET is not None,'Dataset targeted v1.1 completo non trovato.';BASE_SOURCE=BASE_DATASET
calibration_candidates=([Path(os.environ['HAYFLOW_CALIBRATION_SOURCE']).expanduser()] if os.environ.get('HAYFLOW_CALIBRATION_SOURCE') else [])+[Path('/kaggle/input/datasets/alessandrobelli/hayflow-dendritic-protocol-calibration/hayflow_dendritic_protocol_calibration')]+[p.parent for p in INPUT_ROOT.rglob('selected_dendritic_protocols.json')];CALIBRATION_SOURCE=next((p.resolve() for p in calibration_candidates if (p/'selected_dendritic_protocols.json').is_file()),None);assert CALIBRATION_SOURCE is not None,'Calibrazione 01b non trovata.'
topup_candidates=([Path(os.environ['HAYFLOW_TOPUP_V3']).expanduser()] if os.environ.get('HAYFLOW_TOPUP_V3') else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')];TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow05jo_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE;manifests=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifests)==1,manifests;COMPOSITE_MANIFEST=manifests[0]
from src.hayflow_model.hines_state_normalization_repair import EXPECTED_05H_INDEX_SHA256
from src.hayflow_model.hines_netcon_semantic_repair import EXPECTED_05I_INDEX_SHA256
from src.hayflow_model.hines_synaptic_domain_repair import EXPECTED_05IB_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_recheck import EXPECTED_05IC_INDEX_SHA256
from src.hayflow_model.hines_repaired_representation_revision import EXPECTED_05J_INDEX_SHA256
from src.hayflow_model.hines_spatial_support_revision import EXPECTED_05JB_INDEX_SHA256
from src.hayflow_model.hines_trainable_topology_canary import EXPECTED_05JC_INDEX_SHA256
from src.hayflow_model.hines_architecture_reassessment import EXPECTED_05JD_INDEX_SHA256
from src.hayflow_model.hines_region_mechanism_experts import EXPECTED_05JE_INDEX_SHA256
from src.hayflow_model.hines_regenerative_state_decomposition import EXPECTED_05JF_INDEX_SHA256
from src.hayflow_model.hines_regenerative_support_expansion import EXPECTED_05JG_INDEX_SHA256
from src.hayflow_model.hines_regenerative_confirmation import EXPECTED_05JH_INDEX_SHA256,EXPECTED_05JI_INDEX_SHA256
from src.hayflow_model.hines_voltage_objective_reassessment import EXPECTED_05JJ_INDEX_SHA256
from src.hayflow_model.hines_residual_safety_gate import EXPECTED_05JK_INDEX_SHA256
from src.hayflow_model.hines_regenerative_decoder_refit import EXPECTED_05JL_INDEX_SHA256,EXPECTED_05JM_INDEX_SHA256
from src.hayflow_model.hines_regenerative_fresh_test import EXPECTED_05JN_INDEX_SHA256
CHECKPOINT_05B_SOURCE=marker_artifact('hayflow_hines_canary_v2.zip','canary_models.pt');CHECKPOINT_05B_SOURCE=CHECKPOINT_05B_SOURCE.parent if CHECKPOINT_05B_SOURCE.name=='checkpoints' else CHECKPOINT_05B_SOURCE
ARTIFACT_05C_SOURCE=marker_artifact('hayflow_hines_causal_isolation.zip','checkpoint_forensics.json');ARTIFACT_05D_SOURCE=marker_artifact('hayflow_hines_residual_conditioning.zip','free_residual_report.json');ARTIFACT_05E_SOURCE=marker_artifact('hayflow_hines_segment_capacity.zip','capacity_probe_report.json');ARTIFACT_05F_SOURCE=marker_artifact('hayflow_hines_segment_micro_canary.zip','micro_canary_report.json');ARTIFACT_05G_SOURCE=marker_artifact('hayflow_hines_optimization_audit.zip','optimization_support.json')
specs=[('05H','hayflow_hines_representation_forensics.zip','representation_forensics_config.json',EXPECTED_05H_INDEX_SHA256),('05I','hayflow_hines_state_normalization_repair.zip','state_normalization_repair_config.json',EXPECTED_05I_INDEX_SHA256),('05IB','hayflow_hines_netcon_semantic_state_repair.zip','netcon_semantic_repair_config.json',EXPECTED_05IB_INDEX_SHA256),('05IC','hayflow_hines_synaptic_domain_repair.zip','synaptic_domain_repair_config.json',EXPECTED_05IC_INDEX_SHA256),('05J','hayflow_hines_repaired_representation_recheck.zip','repaired_representation_recheck_config.json',EXPECTED_05J_INDEX_SHA256),('05JB','hayflow_hines_repaired_representation_revision.zip','repaired_representation_revision_config.json',EXPECTED_05JB_INDEX_SHA256),('05JC','hayflow_hines_spatial_support_revision.zip','spatial_support_revision_config.json',EXPECTED_05JC_INDEX_SHA256),('05JD','hayflow_hines_trainable_topology_decoder_micro_canary.zip','trainable_topology_canary_config.json',EXPECTED_05JD_INDEX_SHA256),('05JE','hayflow_hines_architecture_reassessment.zip','architecture_reassessment_config.json',EXPECTED_05JE_INDEX_SHA256),('05JF','hayflow_hines_region_mechanism_expert_revision.zip','region_mechanism_expert_config.json',EXPECTED_05JF_INDEX_SHA256),('05JG','hayflow_hines_regenerative_state_decomposition.zip','state_target_decomposition_config.json',EXPECTED_05JG_INDEX_SHA256),('05JH','hayflow_hines_regenerative_support_expansion.zip','regenerative_support_expansion_config.json',EXPECTED_05JH_INDEX_SHA256),('05JI','hayflow_regenerative_confirmation_support.zip','confirmation_plan.json',EXPECTED_05JI_INDEX_SHA256),('05JJ','hayflow_hines_regenerative_confirmation.zip','independent_confirmation_config.json',EXPECTED_05JJ_INDEX_SHA256),('05JK','hayflow_hines_voltage_objective_reassessment.zip','voltage_objective_reassessment_config.json',EXPECTED_05JK_INDEX_SHA256),('05JL','hayflow_hines_residual_safety_gate.zip','residual_safety_gate_config.json',EXPECTED_05JL_INDEX_SHA256),('05JM','hayflow_regenerative_training_support.zip','acquisition_contract.json',EXPECTED_05JM_INDEX_SHA256),('05JN','hayflow_hines_regenerative_decoder_refit.zip','regenerative_decoder_refit_config.json',EXPECTED_05JN_INDEX_SHA256)]
for key,name,marker,expected in specs:globals()[f'ARTIFACT_{key}_SOURCE']=artifact(f'HAYFLOW_{key}_ARTIFACT',name,marker,expected)
print({'base':str(BASE_DATASET),'calibration':str(CALIBRATION_SOURCE),'05j-m':str(ARTIFACT_05JM_SOURCE),'05j-n':str(ARTIFACT_05JN_SOURCE)})

## 4. Autorizzazione, apertura del piano e snapshot fresh-test

In [ ]:
from IPython.display import display
from src.hayflow_teacher import RegenerativeFreshTestConfig,RegenerativeFreshTestSession,expected_audit_hashes
fresh_payload=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_regenerative_fresh_test.yml').read_text());fresh_teacher_config=RegenerativeFreshTestConfig.from_mapping(fresh_payload['regenerative_fresh_test']);TARGET_CONFIG_PATH=ELM_REPO/'configs/hayflow/targeted_transition_dataset_v1_1.yml';target_config=yaml.safe_load(TARGET_CONFIG_PATH.read_text())
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_hines_regenerative_fresh_test');assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova.'
teacher_session=RegenerativeFreshTestSession(ELM_REPO,TEACHER_REPO,base_dataset=BASE_DATASET,calibration_source=CALIBRATION_SOURCE,dataset_config_path=TARGET_CONFIG_PATH,output_dir=OUTPUT_DIR,seed=314159,expected_teacher_hashes=expected_audit_hashes(),native_snapshot_stride=target_config['storage']['native_snapshot_stride_ms'],artifact_05jm_source=ARTIFACT_05JM_SOURCE,artifact_05jn_source=ARTIFACT_05JN_SOURCE,fresh_test_config=fresh_teacher_config)
teacher_report=teacher_session.prepare_teacher();contract_report=teacher_session.prepare_targeted_contract();base_report=teacher_session.verify_base_dataset(verify_large_hdf=True);equilibrium_report=teacher_session.import_base_equilibrium();fresh_protocols,authorization=teacher_session.open_authorized_plan();snapshot_report=teacher_session.prepare_snapshot_bank(fresh_protocols,conditioning_ms=fresh_teacher_config.conditioning_ms)
# Non mostrare snapshot_report['snapshots']: contiene migliaia di valori RNG e può bloccare il browser Kaggle.
display({'teacher_segments':teacher_report['segment_count'],'base_valid':base_report['valid'],'authorization':{'valid':authorization['valid'],'protocol_plan_sha256':authorization['protocol_plan_sha256'],'pair_count':authorization['pair_count'],'episode_count':authorization['episode_count'],'transition_count':authorization['transition_count'],'seed_range':[authorization['seed_start'],authorization['seed_end']],'05jn_verified_before_plan_parsing':authorization['05jn_verified_before_plan_parsing']},'snapshot_bank':{'valid':snapshot_report['valid'],'snapshot_count':snapshot_report['snapshot_count'],'conditioning_ms':snapshot_report['conditioning_ms'],'split_specific':snapshot_report['split_specific']}});assert teacher_report['segment_count']==642 and base_report['valid'] and authorization['valid'];assert snapshot_report['snapshot_count']==32 and authorization['05jn_verified_before_plan_parsing']

## 5. Generazione e replay esaustivo dei 32 pair

In [ ]:
fresh_manifest=teacher_session.generate_fresh_test_shard(fresh_protocols);teacher_fresh_report=teacher_session.validate_fresh_test_shard(fresh_protocols)
replay=teacher_fresh_report['exhaustive_replay'];display({'manifest':{'trajectories':fresh_manifest['trajectory_count'],'transitions':fresh_manifest['transition_count']},'validation':{'valid':teacher_fresh_report['valid'],'pair_contract_valid':teacher_fresh_report['pair_contract_valid'],'transition_store_sha256':teacher_fresh_report['transition_store_sha256'],'generated_splits':teacher_fresh_report['generated_splits'],'seed_range':[teacher_fresh_report['generated_seed_start'],teacher_fresh_report['generated_seed_end']],'replay':{'valid':replay['valid'],'replayed_transition_count':replay.get('replayed_transition_count'),'failure_count':replay.get('failure_count'),'maximum_error':replay.get('maximum_error'),'tolerance':replay.get('tolerance')}}});assert fresh_manifest['trajectory_count']==64 and fresh_manifest['transition_count']==768;assert teacher_fresh_report['valid'] and replay['valid']

## 6. Dataset composito e sessione di valutazione congelata

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle
started,last={},{}
def progress(name,done,total):
 now=time.monotonic();started.setdefault(name,now);pct=int(100*done/total)
 if pct>=last.get(name,-5)+5 or done==total:
  rate=done/max(now-started[name],1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 05j-o][SHA-256 {name}] {pct}% ETA {eta/60:.1f} min',flush=True);last[name]=pct
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=progress);assert bundle.manifest['valid'] and bundle.transition_count==29880
from src.hayflow_model import *
def config(name):return yaml.safe_load((ELM_REPO/'configs/hayflow'/name).read_text())
b=config('hayflow_hines_optimization_audit.yml');f=config('hayflow_hines_representation_forensics.yml');r=config('hayflow_hines_state_normalization_repair.yml');n=config('hayflow_hines_netcon_semantic_repair.yml');d=config('hayflow_hines_synaptic_domain_repair.yml');rc=config('hayflow_hines_repaired_representation_recheck.yml');rv=config('hayflow_hines_repaired_representation_revision.yml');sp=config('hayflow_hines_spatial_support_revision.yml');tp=config('hayflow_hines_trainable_topology_canary.yml');ra=config('hayflow_hines_architecture_reassessment.yml');ex=config('hayflow_hines_region_mechanism_experts.yml');dc=config('hayflow_hines_regenerative_state_decomposition.yml');se=config('hayflow_hines_regenerative_support_expansion.yml');cf=config('hayflow_hines_regenerative_confirmation.yml');oa=config('hayflow_hines_voltage_objective_reassessment.yml');sg=config('hayflow_hines_residual_safety_gate.yml');rf=config('hayflow_hines_regenerative_decoder_refit.yml')
model_config=HinesPrototypeExperimentConfig.from_mapping(b['model_experiment']);isolation_config=HinesIsolationConfig.from_mapping(b['isolation']);conditioning_config=HinesConditioningConfig.from_mapping(b['conditioning']);capacity_config=HinesCapacityConfig.from_mapping(b['capacity']);canary_config=HinesSegmentCanaryConfig.from_mapping(b['micro_canary']);audit_config=HinesOptimizationAuditConfig.from_mapping(b['optimization_audit']);representation_config=HinesRepresentationForensicsConfig.from_mapping(f['representation_forensics']);repair_config=HinesStateNormalizationRepairConfig.from_mapping(r['state_normalization_repair']);netcon_config=HinesNetConSemanticRepairConfig.from_mapping(n['netcon_semantic_repair']);domain_config=HinesSynapticDomainRepairConfig.from_mapping(d['synaptic_domain_repair']);recheck_config=HinesRepairedRepresentationRecheckConfig.from_mapping(rc['repaired_representation_recheck']);revision_config=HinesRepairedRepresentationRevisionConfig.from_mapping(rv['repaired_representation_revision']);spatial_config=HinesSpatialSupportRevisionConfig.from_mapping(sp['spatial_support_revision']);topology_config=HinesTrainableTopologyCanaryConfig.from_mapping(tp['trainable_topology_canary']);reassessment_config=HinesArchitectureReassessmentConfig.from_mapping(ra['architecture_reassessment']);expert_config=HinesRegionMechanismExpertConfig.from_mapping(ex['region_mechanism_experts']);decomposition_config=HinesRegenerativeStateDecompositionConfig.from_mapping(dc['regenerative_state_decomposition']);support_expansion_config=HinesRegenerativeSupportExpansionConfig.from_mapping(se['regenerative_support_expansion']);confirmation_config=HinesRegenerativeConfirmationConfig.from_mapping(cf['independent_confirmation']);objective_config=HinesVoltageObjectiveReassessmentConfig.from_mapping(oa['voltage_objective_reassessment']);safety_config=HinesResidualSafetyGateConfig.from_mapping(sg['residual_safety_gate']);refit_config=HinesRegenerativeDecoderRefitConfig.from_mapping(rf['regenerative_decoder_refit']);fresh_model_config=HinesRegenerativeFreshTestConfig.from_mapping(fresh_payload['frozen_model_gate'])
session=HinesRegenerativeFreshTestEvaluation(bundle,OUTPUT_DIR,model_config,isolation_config,conditioning_config,capacity_config,canary_config,audit_config,representation_config,CHECKPOINT_05B_SOURCE,ARTIFACT_05C_SOURCE,ARTIFACT_05D_SOURCE,ARTIFACT_05E_SOURCE,ARTIFACT_05F_SOURCE,ARTIFACT_05G_SOURCE,repair_config=repair_config,artifact_05h_source=ARTIFACT_05H_SOURCE,netcon_config=netcon_config,artifact_05i_source=ARTIFACT_05I_SOURCE,domain_config=domain_config,artifact_05ib_source=ARTIFACT_05IB_SOURCE,recheck_config=recheck_config,artifact_05ic_source=ARTIFACT_05IC_SOURCE,revision_config=revision_config,artifact_05j_source=ARTIFACT_05J_SOURCE,spatial_config=spatial_config,artifact_05jb_source=ARTIFACT_05JB_SOURCE,topology_config=topology_config,artifact_05jc_source=ARTIFACT_05JC_SOURCE,reassessment_config=reassessment_config,artifact_05jd_source=ARTIFACT_05JD_SOURCE,expert_config=expert_config,artifact_05je_source=ARTIFACT_05JE_SOURCE,decomposition_config=decomposition_config,artifact_05jf_source=ARTIFACT_05JF_SOURCE,support_expansion_config=support_expansion_config,artifact_05jg_source=ARTIFACT_05JG_SOURCE,confirmation_config=confirmation_config,artifact_05jh_source=ARTIFACT_05JH_SOURCE,artifact_05ji_source=ARTIFACT_05JI_SOURCE,objective_config=objective_config,artifact_05jj_source=ARTIFACT_05JJ_SOURCE,safety_config=safety_config,artifact_05jk_source=ARTIFACT_05JK_SOURCE,refit_config=refit_config,artifact_05jl_source=ARTIFACT_05JL_SOURCE,artifact_05jm_source=ARTIFACT_05JM_SOURCE,fresh_test_config=fresh_model_config,artifact_05jn_source=ARTIFACT_05JN_SOURCE,fresh_dataset_root=OUTPUT_DIR,code_revision=REVISION)
prepare_report=session.prepare_fresh_test_evaluation();display({'fresh_store':prepare_report['fresh_store'],'05j-n':prepare_report['artifact_05jn']});assert prepare_report['fresh_store']['valid'] and not prepare_report['checkpoint_selection_performed'] and not prepare_report['retraining_performed']

## 7. Ricostruzione della rappresentazione 05j-n congelata

In [ ]:
session.apply_verified_synaptic_domain_normalizer();session.build_expanded_train_support();session.prepare_expanded_spatial_features();design=session.prepare_topology_canary_designs();session.fit_fixed_tree_ridge_baseline();reconstruction=session.reconstruct_frozen_checkpoints(metric_atol=expert_config.checkpoint_reconstruction_metric_atol);session.build_regenerative_support();session.prepare_expanded_regenerative_roles();expanded=session.reconstruct_expanded_direct_tree_ensemble();external=session.prepare_external_confirmation_roles();role_report=session.prepare_refit_roles();display({'design':design['valid'],'reconstruction':reconstruction['valid'],'expanded':expanded['valid'],'development':external['valid'],'refit_roles':role_report['valid']});assert design['valid'] and reconstruction['valid'] and expanded['valid'] and external['valid'] and role_report['valid']

## 8. Inferenza fresh-test e decisione

In [ ]:
evaluation_report=session.evaluate_frozen_checkpoints();display(pd.DataFrame([{'seed':r['seed'],'rmse_mv':r['metrics']['aggregate_voltage_rmse_mv'],'max_error_mv':r['metrics']['maximum_segment_error_mv'],'branching_retention':r['metrics']['median_branching_retention'],'improvement':r['improvement_vs_best_baseline_fraction'],'passed':r['run_passed']} for r in evaluation_report['runs']]));display({'baselines':{k:v['aggregate_voltage_rmse_mv'] for k,v in evaluation_report['baselines'].items()},'ensemble_rmse':evaluation_report['ensemble_mean_metrics']['aggregate_voltage_rmse_mv'],'passing_seeds':evaluation_report['passing_seed_count'],'gate':evaluation_report['robust_fresh_test_gate_passed']});assert evaluation_report['all_pairs_evaluated'] and not evaluation_report['checkpoint_selection_performed'] and not evaluation_report['retraining_performed']
final_report=session.finalize_fresh_test(teacher_fresh_report,evaluation_report);display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'candidate':final_report['candidate_model_authorized'],'micro_rollout':final_report['micro_rollout_authorized'],'full_training':final_report['full_training_authorized'],'next_step':final_report['next_step']});assert final_report['valid'] and not final_report['full_training_authorized']

## 9. Crea e scarica lo ZIP

In [ ]:
from shutil import make_archive
import base64
from IPython.display import Javascript,display
zip_path=Path(make_archive('/kaggle/working/hayflow_hines_regenerative_fresh_test','zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(zip_path.read_bytes()).decode('ascii');filename=zip_path.name
display(Javascript(f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{filename}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""));print({'zip':str(zip_path),'size_mib':round(zip_path.stat().st_size/2**20,2),'download':'avviato dal browser'})